# Strategy Research Template

## Hypothesis
_Describe the alpha hypothesis here._

## Data
_Describe the data sources and time period._

## EDA
_Exploratory data analysis._

## Statistical Tests
_Validate assumptions: stationarity, cointegration, etc._

## Feature Engineering
_Features used in the model._

## Model
_Model description and parameters._

## Results
_Backtest performance._

## Interpretation
_What do results mean? Is alpha real or spurious?_

## Failure Analysis
_Where does the strategy fail? Regime dependence?_

In [ ]:
import sys
sys.path.insert(0, '../../')

from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

from python.market_data import YFinanceProvider
from python.feature_engineering import add_all_features
from python.statistical_research import hurst_exponent, adf_test
from python.performance_analytics import compute_metrics, print_tearsheet

In [ ]:
# ---- 1. Download Data ----
provider = YFinanceProvider()
symbols = ['SPY', 'QQQ', 'IWM']
start = datetime(2020, 1, 1)
end   = datetime(2024, 12, 31)

data = provider.fetch_multiple(symbols, start, end, resolution='1d')
for sym, df in data.items():
    print(f"{sym}: {len(df)} bars  ({df.index.min().date()} -> {df.index.max().date()})")

In [ ]:
# ---- 2. Feature Engineering ----
featured = {sym: add_all_features(df) for sym, df in data.items()}
featured['SPY'].tail()

In [ ]:
# ---- 3. EDA ----
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for i, sym in enumerate(symbols):
    data[sym]['close'].plot(ax=axes[i], title=sym)
plt.tight_layout()

In [ ]:
# ---- 4. Statistical Tests ----
for sym, df in data.items():
    h   = hurst_exponent(df['close'])
    ret = df['close'].pct_change().dropna()
    adf = adf_test(ret)
    print(f"{sym}:  Hurst={h:.3f}  ADF stationary={adf['stationary']}  p={adf['pvalue']:.4f}")

In [ ]:
# ---- 5. Strategy Signals (placeholder) ----
from python.strategy_research import MeanReversionStrategy

strategy = MeanReversionStrategy(window=20, z_entry=2.0)
signals = strategy.generate_signals(data)
print('Current signals:', signals)

In [ ]:
# ---- 6. Simple Return-Based Backtest ----
returns = data['SPY']['close'].pct_change().dropna()
metrics = compute_metrics(returns)
print_tearsheet(metrics)